In [84]:
!pip install -q openai

In [85]:
from openai import OpenAI
from google.colab import files
import json
from copy import deepcopy
from IPython.display import Audio, display
import os

In [86]:
API_KEY = "sk-UnH0nt25sjFOMakgdLhpOw"
BASE_URL = "https://elmodels.ngrok.app/v1"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

In [87]:
ACTIVITY_LEVEL_OPTIONS = ["قليل", "متوسط", "عالي"]

GOAL_OPTIONS = [
    "إنقاص الوزن",
    "تحسين اللياقة",
    "زيادة النشاط",
    "الحفاظ على الصحة",
    "أخرى"
]

DAYS_PER_WEEK_OPTIONS = [
    "2–3 أيام",
    "3–4 أيام",
    "5 أيام أو أكثر"
]

SLEEP_QUALITY_OPTIONS = ["جيد", "متوسط", "سيئ"]

FATIGUE_LEVEL_OPTIONS = ["خفيف", "متوسط", "شديد"]

INJURY_HISTORY_OPTIONS = ["نعم", "لا"]

CURRENT_SYMPTOMS_OPTIONS = [
    "لا يوجد",
    "دوخة",
    "ضيق تنفس",
    "ألم حاد في العضلات",
    "أخرى"
]

MENSTRUAL_STATUS_OPTIONS = [
    "نعم، مع ألم",
    "نعم، بدون ألم",
    "لا"
]

ENVIRONMENT_PREFERENCE_OPTIONS = [
    "داخلية",
    "خارجية",
    "لا فرق"
]

WEATHER_CONDITION_OPTIONS = [
    "معتدل",
    "حار",
    "مشمس",
    "جاف",
    "رطب",
    "مغبر"
]

In [88]:
def create_profile_data(
    age,
    height,
    weight,
    activity_level,
    goals,
    goals_other,
    days_per_week,
    injury,
    injury_details
):
    return {
        "age": age,
        "height": height,
        "weight": weight,
        "activity_level": activity_level,
        "goals": goals,
        "goals_other": goals_other,
        "days_per_week": days_per_week,
        "injury": injury,
        "injury_details": injury_details
    }

def create_daily_checkin(
    sleep,
    fatigue,
    symptoms,
    symptoms_other,
    menstrual,
    environment,
    weather
):
    return {
        "sleep": sleep,
        "fatigue": fatigue,
        "symptoms": symptoms,
        "symptoms_other": symptoms_other,
        "menstrual": menstrual,
        "environment": environment,
        "weather": weather
    }

In [89]:
RISK_WEIGHTS = {
    "sleep": {
        "جيد": 0,
        "متوسط": 1,
        "سيئ": 2
    },

    "fatigue": {
        "خفيف": 0,
        "متوسط": 1,
        "شديد": 2
    },

    "symptoms": {
        "لا يوجد": 0,
        "دوخة": 2,
        "ضيق تنفس": 3,
        "ألم حاد في العضلات": 2,
        "أخرى": 1
    },

    "menstrual": {
        "لا": 0,
        "نعم، بدون ألم": 1,
        "نعم، مع ألم": 2
    },

    "weather": {
        "معتدل": 0,
        "مشمس": 1,
        "حار": 2,
        "جاف": 1,
        "رطب": 1,
        "مغبر": 2
    },

    "injury": {
        "لا": 0,
        "نعم": 2
    }
}

In [90]:
def risk_engine(user):
    risk_score = 0
    flags = []

    # النوم
    sleep_score = RISK_WEIGHTS["sleep"][user["sleep"]]
    risk_score += sleep_score
    if sleep_score > 0:
        flags.append("نوم غير كافٍ")

    # التعب
    fatigue_score = RISK_WEIGHTS["fatigue"][user["fatigue"]]
    risk_score += fatigue_score
    if fatigue_score > 0:
        flags.append("إجهاد")

    # الأعراض
    for sym in user["symptoms"]:
        if sym != "لا يوجد" and sym in RISK_WEIGHTS["symptoms"]:
            sym_score = RISK_WEIGHTS["symptoms"][sym]
            risk_score += sym_score
            if sym_score > 0:
                flags.append(sym)

    # إذا كتبت المستخدمة عرضًا إضافيًا
    if user.get("symptoms_other", "").strip():
        risk_score += 1
        flags.append("أعراض إضافية مذكورة")

    # الدورة
    menstrual_score = RISK_WEIGHTS["menstrual"][user["menstrual"]]
    risk_score += menstrual_score
    if menstrual_score > 0:
        flags.append("حالة متعلقة بالدورة")

    # الجو
    weather_value = user["weather"]

    if isinstance(weather_value, list):
        for w in weather_value:
            if w in RISK_WEIGHTS["weather"]:
                weather_score = RISK_WEIGHTS["weather"][w]
                risk_score += weather_score
                if weather_score > 0:
                    flags.append(f"الجو: {w}")
    else:
        weather_score = RISK_WEIGHTS["weather"][weather_value]
        risk_score += weather_score
        if weather_score > 0:
            flags.append("ظروف جوية تحتاج مراعاة")

    # الإصابة
    injury_score = RISK_WEIGHTS["injury"][user["injury"]]
    risk_score += injury_score
    if injury_score > 0:
        flags.append("إصابة سابقة")

    return {
        "risk_score": risk_score,
        "flags": list(set(flags))
    }

In [91]:
def decision_engine(user, risk):
    risk_score = risk["risk_score"]

    # تحديد مستوى الخطر
    if risk_score >= 8:
        risk_level = "مرتفع"
    elif risk_score >= 4:
        risk_level = "متوسط"
    else:
        risk_level = "منخفض"

    # المستوى الأساسي
    if user["activity_level"] == "عالي":
        base_level = "متقدم"
    elif user["activity_level"] == "متوسط":
        base_level = "متوسط"
    else:
        base_level = "مبتدئ"

    level = base_level
    intensity = "طبيعية"

    # تعديل المستوى والشدة حسب مستوى الخطر
    if risk_level == "مرتفع":
        level = "مبتدئ"
        intensity = "خفيفة جدًا"
    elif risk_level == "متوسط":
        if base_level == "متقدم":
            level = "متوسط"
        else:
            level = "مبتدئ"
        intensity = "خفيفة"
    else:
        intensity = "طبيعية"

    # نوع الخطة حسب الهدف
    goals = user["goals"]

    if "إنقاص الوزن" in goals:
        plan_type = "كارديو خفيف"
    elif "تحسين اللياقة" in goals:
        plan_type = "تمارين مقاومة"
    elif "زيادة النشاط" in goals:
        plan_type = "تمارين حركة خفيفة"
    else:
        plan_type = "تمارين خفيفة"

    # تعديل نوع الخطة حسب الحالة
    if user["injury"] == "نعم":
        plan_type = "تمارين مخصصة بدون ضغط"

    if user["menstrual"] == "نعم، مع ألم":
        plan_type = "تمارين تمدد واسترخاء"

    if "دوخة" in user["symptoms"] or "ضيق تنفس" in user["symptoms"]:
        plan_type = "راحة أو حركة خفيفة جدًا"

    # تحديد البيئة
    weather_value = user["weather"]
    bad_weather = ["حار", "مغبر", "رطب"]

    if isinstance(weather_value, list):
        has_bad_weather = any(w in bad_weather for w in weather_value)
    else:
        has_bad_weather = weather_value in bad_weather

    if user["environment"] == "داخلية":
        environment = "تمرين داخلي"
    elif has_bad_weather:
        environment = "تمرين داخلي"
    else:
        environment = "تمرين خارجي"

    # تحديد المدة
    if risk_level == "مرتفع":
        if "دوخة" in user["symptoms"] or "ضيق تنفس" in user["symptoms"]:
            duration = "5–10 دقائق أو راحة"
        else:
            duration = "10–15 دقيقة"
    elif risk_level == "متوسط":
        duration = "15–25 دقيقة"
    else:
        if user["days_per_week"] == "2–3 أيام":
            duration = "20–30 دقيقة"
        elif user["days_per_week"] == "3–4 أيام":
            duration = "30–40 دقيقة"
        else:
            duration = "40–60 دقيقة"

    return {
        "base_level": base_level,
        "level": level,
        "risk_level": risk_level,
        "intensity": intensity,
        "environment": environment,
        "plan_type": plan_type,
        "duration": duration,
        "risk_flags": risk["flags"]
    }

In [92]:
def build_prompt(user, decision):
    return f"""
أنتِ مساعدة صحية ذكية تعملين داخل تطبيق اسمه "وثبه"،
وهو تطبيق مخصص لدعم المرأة في رحلتها الصحية اليومية.

يعرض التطبيق للمستخدمة النتائج التالية فقط:
- المستوى
- نوع الخطة
- المدة
- الشدة
- التوصية اليومية
- تنبيه السلامة
- ملاحظات البيئة
- رسالة تحفيزية

مهم:
- استخدمي بيانات المستخدمة ونتائج التحليل المتاحة لك
- أنشئي ناتجًا مناسبًا للعرض داخل بطاقات قصيرة وواضحة
- اجعلي النصوص مناسبة للعرض على الجوال
- لا تكتبي أي شرح خارج JSON

بيانات المستخدمة:
{json.dumps(user, ensure_ascii=False, indent=2)}

نتائج التحليل:
{json.dumps(decision, ensure_ascii=False, indent=2)}

التعليمات:
1. level:
   - حددي المستوى النهائي المناسب للمستخدمة

2. plan_type:
   - نوع الخطة المناسب مثل:
     "تمارين منزلية خفيفة"
     "تمارين مقاومة خفيفة"
     "تمارين كارديو معتدلة"

3. duration:
   - مدة واضحة ومختصرة

4. intensity:
   - الشدة المناسبة للحالة اليومية

5. recommendation:
   - اكتبي التوصية اليومية الرئيسية
   - ادمجي معها التمارين أو الأنشطة المقترحة داخل نفس النص
   - اجعليها عملية وقابلة للتنفيذ
   - لا تضعي قائمة منفصلة للتمارين خارج recommendation

6. safety_note:
   - اكتبي تنبيه السلامة
   - ادمجي داخله أي مخاطر أو تحذيرات مهمة مثل:
     التعب، الإصابة، الأعراض، الدورة، أو ضرورة التوقف

7. environment_note:
   - لا تكتبي البيئة كخانة مستقلة
   - ادمجيها داخل النص نفسه
   - مثال:
     "يُفضل التمرين الداخلي اليوم بسبب الجو الحار"
     أو
     "يمكنك أداء التمرين خارج المنزل لأن الجو معتدل"

8. motivation:
   - رسالة تحفيزية قصيرة وواضحة

قواعد مهمة:
- إذا كانت الأعراض تشمل دوخة أو ضيق تنفس، فالأولوية للسلامة
- إذا كان التعب شديدًا أو النوم سيئًا، خففي الشدة
- إذا كانت الدورة مع ألم، اجعلي الخطة ألطف
- إذا كان الجو حارًا أو مغبرًا أو رطبًا، فضلي التمرين الداخلي
- اجعلي النصوص مختصرة، واضحة، ومناسبة للواجهة

أعيدي النتيجة بصيغة JSON فقط بهذا الشكل:

{{
  "level": "",
  "plan_type": "",
  "duration": "",
  "intensity": "",
  "recommendation": "",
  "safety_note": "",
  "environment_note": "",
  "motivation": "",
  "risk_level": ""
}}
"""

In [93]:
def generate_response(user, decision):
    prompt = build_prompt(user, decision)

    response = client.chat.completions.create(
        model="nuha-2.0",
        messages=[
            {
                "role": "system",
                "content": "أنتِ نظام توصية صحي ذكي. التزمي بإخراج JSON فقط."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    raw_text = response.choices[0].message.content
    return json.loads(raw_text)

In [94]:
def print_result(result):
    print("المستوى:", result["level"])
    print("مستوى الخطر:", result["risk_level"])
    print("نوع الخطة:", result["plan_type"])
    print("المدة:", result["duration"])
    print("الشدة:", result["intensity"])
    print("\nالتوصية اليومية:")
    print(result["recommendation"])
    print("\nتنبيه السلامة:")
    print(result["safety_note"])
    print("\nملاحظات البيئة:")
    print(result["environment_note"])
    print("\nرسالة تحفيزية:")
    print(result["motivation"])

In [95]:
test_cases = [
    {
        "case_name": "حالة طبيعية",
        "profile": create_profile_data(
            age=22,
            height=160,
            weight=60,
            activity_level="متوسط",
            goals=["تحسين اللياقة"],
            goals_other="",
            days_per_week="3–4 أيام",
            injury="لا",
            injury_details=""
        ),
        "daily": create_daily_checkin(
            sleep="جيد",
            fatigue="خفيف",
            symptoms=["لا يوجد"],
            symptoms_other="",
            menstrual="لا",
            environment="خارجية",
            weather="معتدل"
        )
    },
    {
        "case_name": "حالة تحتاج تخفيف",
        "profile": create_profile_data(
            age=24,
            height=158,
            weight=68,
            activity_level="قليل",
            goals=["زيادة النشاط"],
            goals_other="",
            days_per_week="2–3 أيام",
            injury="لا",
            injury_details=""
        ),
        "daily": create_daily_checkin(
            sleep="متوسط",
            fatigue="شديد",
            symptoms=["لا يوجد"],
            symptoms_other="",
            menstrual="نعم، مع ألم",
            environment="داخلية",
            weather="حار"
        )
    },
    {
        "case_name": "حالة عالية الخطورة",
        "profile": create_profile_data(
            age=27,
            height=165,
            weight=72,
            activity_level="عالي",
            goals=["إنقاص الوزن"],
            goals_other="",
            days_per_week="5 أيام أو أكثر",
            injury="نعم",
            injury_details="إصابة في الركبة"
        ),
        "daily": create_daily_checkin(
            sleep="سيئ",
            fatigue="شديد",
            symptoms=["دوخة"],
            symptoms_other="",
            menstrual="لا",
            environment="خارجية",
            weather="حار"
        )
    }
]

In [96]:
for case in test_cases:
    print("=" * 80)
    print("اسم الحالة:", case["case_name"])

    user = {**case["profile"], **case["daily"]}

    risk = risk_engine(user)
    decision = decision_engine(user, risk)
    result = generate_response(user, decision)

    print("\nالمدخلات:")
    print(json.dumps(user, ensure_ascii=False, indent=2))

    print("\nتحليل المخاطر:")
    print(json.dumps(risk, ensure_ascii=False, indent=2))

    print("\nقرار النظام:")
    print(json.dumps(decision, ensure_ascii=False, indent=2))

    print("\nالناتج النهائي:")
    print(json.dumps(result, ensure_ascii=False, indent=2))

    print("\nعرض مبسط:")
    print_result(result)
    print("\n")

اسم الحالة: حالة طبيعية

المدخلات:
{
  "age": 22,
  "height": 160,
  "weight": 60,
  "activity_level": "متوسط",
  "goals": [
    "تحسين اللياقة"
  ],
  "goals_other": "",
  "days_per_week": "3–4 أيام",
  "injury": "لا",
  "injury_details": "",
  "sleep": "جيد",
  "fatigue": "خفيف",
  "symptoms": [
    "لا يوجد"
  ],
  "symptoms_other": "",
  "menstrual": "لا",
  "environment": "خارجية",
  "weather": "معتدل"
}

تحليل المخاطر:
{
  "risk_score": 0,
  "flags": []
}

قرار النظام:
{
  "base_level": "متوسط",
  "level": "متوسط",
  "risk_level": "منخفض",
  "intensity": "طبيعية",
  "environment": "تمرين خارجي",
  "plan_type": "تمارين مقاومة",
  "duration": "30–40 دقيقة",
  "risk_flags": []
}

الناتج النهائي:
{
  "level": "متوسط",
  "plan_type": "تمارين مقاومة",
  "duration": "30–40 دقيقة",
  "intensity": "طبيعية",
  "recommendation": "ابدئي بتمرين خارجي يركز على تمارين المقاومة لتحسين اللياقة، مع التركيز على مجموعات عضلية مختلفة لمدة 30-40 دقيقة.",
  "safety_note": "بما أن مستوى التعب خفيف ولا ت

In [100]:
tts_text = "هذه خطتك اليوم. ابدئي بتمارين خفيفة داخل المنزل، مع التركيز على الراحة والتنفس."

tts_response = client.audio.speech.create(
    model="elm-tts",
    voice="female",   # ← هذا المطلوب
    input=tts_text
)

In [101]:
output_audio_path = "tts_output.mp3"

with open(output_audio_path, "wb") as f:
    f.write(tts_response.read())

print("تم إنشاء الملف الصوتي:", output_audio_path)

تم إنشاء الملف الصوتي: tts_output.mp3


In [102]:
display(Audio(output_audio_path))

In [103]:
tts_text = f"""
خطتك اليوم:
{result['recommendation']}

تنبيه السلامة:
{result['safety_note']}

ملاحظات البيئة:
{result['environment_note']}
"""

tts_response = client.audio.speech.create(
    model="elm-tts",
    voice="female",
    input=tts_text
)

output_audio_path = "plan_audio.mp3"
tts_response.stream_to_file(output_audio_path)

from IPython.display import Audio
Audio(output_audio_path)

/tmp/ipykernel_9206/608638616.py:19: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  tts_response.stream_to_file(output_audio_path)
